# CausalIF Auto MPG Demo

**Business question: What factors influence MPG in cars?**

This self-contained notebook demonstrates the [CausalIF](https://github.com/awslabs/causalif) framework (AWS Labs) on the [UCI Auto MPG dataset](https://archive.ics.uci.edu/dataset/9/auto+mpg). It acquires and prepares the data, runs CausalIF causal discovery to identify which vehicle attributes causally affect fuel efficiency (miles per gallon), and visualizes the resulting causal graph to answer the business question above.

All executable code lives in this single notebook. It is intended to run in **AWS SageMaker Studio**, where AWS credentials for Amazon Bedrock are provided implicitly by the Studio execution role.

## Prerequisites

You do **not** need to create an S3 bucket or a knowledge base by hand — the first code cell (Section 0, *Automated setup*) does that for you in your own AWS account. You only need the right permissions on the SageMaker execution role:

1. **Amazon Bedrock model access.** Serverless foundation models auto-enable on first invocation (the old "Model access" enablement page is retired), so there is no manual activation step. The SageMaker execution role still needs the `bedrock:InvokeModel` permission for the configured model (`BEDROCK_MODEL_ID`) — covered by the setup permissions below — and it must not be blocked by an IAM policy or SCP. Note: for Anthropic Claude models, a first-time user may be asked to submit brief use-case details before the first call succeeds. CausalIF uses this Bedrock model as its LLM via LangChain `ChatBedrockConverse`; credentials come from the execution role automatically and no access keys are entered anywhere in this notebook.
2. **A region that supports managed knowledge bases.** The default `AWS_REGION` is `us-west-2` (Oregon), which supports both Amazon Bedrock managed knowledge bases and the default Claude model. Managed knowledge bases are available only in select regions (for example `us-east-1`, `us-west-2`, `eu-west-1`) and are **not** available in `us-west-1`.
3. **Permissions for the automated setup.** The execution role must be allowed to create and use the demo resources: `s3:CreateBucket`/`PutObject`/`GetObject`/`ListBucket`, `iam:CreateRole`/`PutRolePolicy`/`GetRole`/`PassRole`, `bedrock:CreateKnowledgeBase`/`CreateDataSource`/`StartIngestionJob`/`GetKnowledgeBase`/`GetIngestionJob`/`ListKnowledgeBases`/`ListDataSources`, `bedrock:Retrieve`, and `bedrock:InvokeModel`. The user guide provides a ready-to-paste inline IAM policy covering all of these. For a quick throwaway demo, attaching broad managed policies (for example `AmazonBedrockFullAccess` plus S3 and IAM access) also works; scope these down for production.

If you already have a knowledge base, set `KNOWLEDGE_BASE_ID` in the Configuration section and the setup cell will skip provisioning. If you set `RUN_BOOTSTRAP = False` and leave `KNOWLEDGE_BASE_ID` blank, the notebook still runs — it just proceeds without a retriever tool and relies on the model's background knowledge alone.

## Notebook structure

The notebook is organized into six ordered sections, executed top-to-bottom. Each stage produces a named variable consumed by the next stage:

1. **Setup** — install pinned dependencies and import the required names.
2. **Configuration** — a single settings cell holding the AWS region, Bedrock model id, automated-setup and Knowledge Base settings, and CausalIF run parameters.
   - **Automated setup (Section 0)** — create an S3 bucket in your account, download the reference documents from the public GitHub repo and upload them to that bucket, and provision a Bedrock *managed* knowledge base, then set `KNOWLEDGE_BASE_ID` automatically (skipped when `RUN_BOOTSTRAP` is `False` or an id is already set).
   - **Retriever tool** — build the managed-knowledge-base retriever tool from the configuration values (skipped automatically when no `KNOWLEDGE_BASE_ID` is set).
3. **Data acquisition** — fetch the UCI Auto MPG dataset into `raw_df`.
4. **Data preparation** — clean and select columns to produce the observational `prepared_df`.
5. **Causal analysis** — configure the CausalIF engine and run causal discovery to produce `result`.
6. **Results presentation** — render the causal graph, an edge table, a written summary, and the plain-language answer to the business question.


## 1. Setup

This first step installs the pinned Python dependencies (CausalIF and its supporting packages) directly into the current SageMaker Studio kernel using the `%pip` magic. Run this cell before any other code in the notebook.

**Expected output:** pip's installation log, ending with a line confirming the packages were installed (for example, `Successfully installed ...`). If the install fails, pip prints the error output in this cell's result and you should resolve it before continuing — the imports live in the next cell so a failed install does not silently proceed. You may need to restart the kernel after a fresh install for the new packages to be importable.

In [ ]:
# Install pinned dependencies into the current kernel.
# A non-zero pip exit surfaces the error output in this cell's result;
# imports are kept in the next cell so a failed install does not proceed silently.
%pip install \
    causalif==0.1.10 \
    langchain==0.3.27 \
    langchain-aws==0.2.35 \
    ucimlrepo==0.0.7 \
    pandas==2.2.3 \
    plotly==5.24.1

In [ ]:
# Imports are kept separate from the install cell above so a failed install
# surfaces its error there and does not silently proceed into a broken import.
# If any import below fails, re-run the install cell (and restart the kernel if
# a fresh install added packages) before continuing.
import json
import time
import uuid
import urllib.request
import pandas as pd
import boto3
from botocore.exceptions import ClientError
from ucimlrepo import fetch_ucirepo
from langchain_aws import ChatBedrockConverse
from langchain_core.tools import Tool
from causalif import (
    set_causalif_engine,
    causalif,
    causalif_intervene,
    visualize_causalif_results,
)


def _require(name, value):
    """Return ``value`` if it is present and non-empty, else raise a ValueError.

    Used by downstream cells to validate configuration variables read from the
    Config_Cell. A missing (None) or empty (empty string / whitespace-only)
    value raises a ValueError naming the offending variable so the notebook
    halts with a clear, actionable message instead of failing obscurely later.
    """
    if value is None:
        raise ValueError(f"Required configuration variable '{name}' is missing (None).")
    if isinstance(value, str) and value.strip() == "":
        raise ValueError(f"Required configuration variable '{name}' is empty.")
    return value

## 2. Configuration

This is the **single settings cell** for the notebook. Every environment-specific value — the AWS region, the Amazon Bedrock model id, the Amazon Bedrock Knowledge Base settings, and the CausalIF run parameters — is defined here and nowhere else. If your AWS environment differs from the defaults (for example, a different region, a Bedrock model your execution role can invoke, or a Knowledge Base id), adjust the variables in the cell below and re-run the notebook top-to-bottom; no other cell needs editing.

**Expected output:** this cell only assigns variables, so it produces no printed output. Downstream cells read these values (and validate the required ones via `_require`) when they run.

In [ ]:
# Config_Cell — the single location for environment-specific settings.
# Adjust these values to match your AWS environment; no other cell redefines
# or hardcodes them. Downstream cells read configuration only from here.

# AWS region used for all Amazon Bedrock calls. Must be a region that supports
# Amazon Bedrock managed knowledge bases AND the Bedrock model below. us-west-2
# (Oregon) supports both. (Managed KBs are NOT available in us-west-1.)
AWS_REGION = "us-west-2"

# Amazon Bedrock model id (non-empty). Default is a US Claude inference profile
# consistent with the default us-west-2 region; change it to a model your
# SageMaker execution role is permitted to invoke.
BEDROCK_MODEL_ID = "us.anthropic.claude-sonnet-4-20250514-v1:0"

# ---------------------------------------------------------------------------
# Automated setup (Section 0 bootstrap) settings.
# The first code cell can create everything the demo needs in YOUR AWS account:
# an S3 bucket, a copy of the reference documents from the shared public demo
# bucket, and a Bedrock MANAGED knowledge base built from those documents.
# ---------------------------------------------------------------------------

# RUN_BOOTSTRAP: when True, the Section 0 cell provisions the bucket + managed
# knowledge base and fills in KNOWLEDGE_BASE_ID for you. Set it to False if you
# already have a knowledge base (fill KNOWLEDGE_BASE_ID below) or want to skip
# the retriever entirely (leave KNOWLEDGE_BASE_ID blank).
RUN_BOOTSTRAP = True

# GITHUB_RAW_BASE / KB_DOC_FILES: the reference documents are downloaded from
# the public GitHub repo and then uploaded into your own S3 bucket (which the
# managed knowledge base ingests). GITHUB_RAW_BASE is the raw.githubusercontent
# base URL of the knowledge-base folder; KB_DOC_FILES lists the files to fetch.
# No AWS credentials are needed to download from GitHub, and nothing is public
# on the S3 side. Point these at a different repo/branch/path if your files
# live elsewhere.
GITHUB_RAW_BASE = (
    "https://raw.githubusercontent.com/awslabs/causalif/main/"
    "examples/auto-mpg/knowledge-base"
)
KB_DOC_FILES = [
    "fuel_economy_primer.md",
    "epa_trends_report.pdf",
]

# TARGET_BUCKET: the bucket created in YOUR account to hold the copied docs.
# Leave as None to auto-generate a globally-unique name of the form
# "causalif-analyticon2026-<uuid>" (recommended so teammates don't collide on
# S3's global namespace). Or set an explicit name you own.
TARGET_BUCKET = None
# TARGET_KB_PREFIX: prefix under TARGET_BUCKET where the docs are copied and
# which the managed knowledge base ingests.
TARGET_KB_PREFIX = "knowledge-base/"

# KB_NAME: name for the managed knowledge base created in your account.
KB_NAME = "causalif-auto-mpg-kb"
# KB_ROLE_NAME: name of the IAM role the knowledge base assumes to read your
# bucket. The bootstrap creates it if it does not already exist.
KB_ROLE_NAME = "CausalIFAutoMpgKBRole"

# Amazon Bedrock Knowledge Base settings (see Prerequisites).
# KNOWLEDGE_BASE_ID: id of a Bedrock knowledge base holding domain reference
# material (column/factor definitions, engineering relationships) for CausalIF
# to retrieve. Leave it blank ("") to let the Section 0 bootstrap create one and
# fill it in automatically. If you set RUN_BOOTSTRAP = False and leave this
# blank, the notebook runs WITHOUT a retriever tool (model background knowledge
# only). If you already have a knowledge base, paste its id here.
KNOWLEDGE_BASE_ID = ""
# KB_NUM_RESULTS: number of passages the retriever returns per query.
KB_NUM_RESULTS = 20
# Name and description advertised to the LLM for the retriever tool. The
# description should tell the model what the knowledge base contains so it
# queries the tool for the right information.
RETRIEVER_TOOL_NAME = "auto_mpg_data_retriever"
RETRIEVER_TOOL_DESCRIPTION = (
    "Searches and returns definitions and domain context for the Auto MPG "
    "vehicle attributes (e.g. mpg, cylinders, displacement, horsepower, "
    "weight, acceleration, model_year)."
)

# CausalIF bootstrap stability parameters.
# BOOTSTRAP_ITERATIONS: number of bootstrap resamples (integer >= 1).
BOOTSTRAP_ITERATIONS = 50
# BOOTSTRAP_THRESHOLD: edge-stability threshold in [0, 1]. Lower is more
# permissive (admits more edges/vertices into the causal graph).
BOOTSTRAP_THRESHOLD = 0.5

# Interventional (do-calculus) analysis settings.
# ENABLE_INTERVENTION drives CausalIF's enable_causal_estimate and the
# interventional section; INTERVENTION_LEVEL is one of "high"/"medium"/"low".
ENABLE_INTERVENTION = True
INTERVENTION_LEVEL = "low"

# Minimum number of prepared records required before CausalIF is run.
MIN_SAMPLES = 100

# Filename for the interactive HTML copy of the causal graph written by the
# results-presentation section (saved next to the notebook).
OUTPUT_HTML = "causalif-mpg-graph.html"

### Automated setup — create the S3 bucket and managed knowledge base

This cell provisions everything the demo needs **in your own AWS account**, so you do not have to create anything by hand in the console. When `RUN_BOOTSTRAP` is `True` (the default) and `KNOWLEDGE_BASE_ID` is still blank, it:

1. **Creates an S3 bucket** in your account (name from `TARGET_BUCKET`, or an auto-generated unique name) in `AWS_REGION`.
2. **Downloads the reference documents from GitHub** (`GITHUB_RAW_BASE`/`KB_DOC_FILES`, public — no AWS credentials needed) and uploads them into your bucket under `TARGET_KB_PREFIX`. Nothing on the S3 side is public.
3. **Creates an IAM role** (`KB_ROLE_NAME`) that the knowledge base assumes to read your bucket and invoke Bedrock.
4. **Creates a Bedrock *managed* knowledge base** (`type=MANAGED`) — Amazon Bedrock manages the vector store, indexing, and embeddings for you, so there is no OpenSearch collection or index to set up.
5. **Attaches an S3 data source and starts an ingestion job**, waiting until it completes.
6. **Sets `KNOWLEDGE_BASE_ID`** to the new knowledge base id so the retriever cell below picks it up automatically.

The cell is **idempotent**: re-running it reuses an existing bucket, role, knowledge base, and data source instead of creating duplicates. If `KNOWLEDGE_BASE_ID` is already set (or `RUN_BOOTSTRAP` is `False`), the cell does nothing and simply reports that it was skipped.

**Required permissions:** the SageMaker execution role running this notebook must be allowed to create/use these resources — `s3:CreateBucket`, `s3:PutObject`/`GetObject`/`ListBucket`, `iam:CreateRole`/`PutRolePolicy`/`PassRole`, and `bedrock:CreateKnowledgeBase`/`CreateDataSource`/`StartIngestionJob`/`GetKnowledgeBase`/`GetIngestionJob`. See the guide's IAM step. For a workshop, an admin-level role is simplest; scope it down for production.

**Expected output:** progress lines for each step ending with a confirmation that the knowledge base is available and `KNOWLEDGE_BASE_ID` has been set. Provisioning the managed knowledge base and ingesting the documents typically takes a few minutes. Any failure prints a step-identifying error and re-raises so you can fix the cause before continuing.

In [ ]:
# Section 0 - Automated setup: S3 bucket + copied docs + managed knowledge base.
# Reads only Config_Cell values. Idempotent and safe to re-run. Sets
# KNOWLEDGE_BASE_ID on success so the retriever cell below uses it.


def _bootstrap_managed_kb():
    """Create bucket + copy docs + managed KB in this account. Returns the KB id."""
    sts = boto3.client("sts", region_name=AWS_REGION)
    account_id = sts.get_caller_identity()["Account"]

    # Resolve the target bucket name. A random UUID suffix keeps it globally
    # unique so teammates never collide on S3's global namespace. The 8-char
    # suffix keeps the name well within S3's 63-character limit.
    target_bucket = TARGET_BUCKET or f"causalif-analyticon2026-{uuid.uuid4().hex[:8]}"

    s3 = boto3.client("s3", region_name=AWS_REGION)
    iam = boto3.client("iam", region_name=AWS_REGION)
    bedrock_agent = boto3.client("bedrock-agent", region_name=AWS_REGION)

    # --- Step 1: create the target bucket (idempotent) ---------------------
    # us-east-1 must NOT send a LocationConstraint; every other region must.
    try:
        if AWS_REGION == "us-east-1":
            s3.create_bucket(Bucket=target_bucket)
        else:
            s3.create_bucket(
                Bucket=target_bucket,
                CreateBucketConfiguration={"LocationConstraint": AWS_REGION},
            )
        print(f"[1/6] Created S3 bucket: {target_bucket}")
    except ClientError as exc:
        _code = exc.response["Error"]["Code"]
        # Already own it (or it exists in this region) -> reuse it.
        if _code in ("BucketAlreadyOwnedByYou", "BucketAlreadyExists"):
            print(f"[1/6] Reusing existing S3 bucket: {target_bucket}")
        else:
            raise

    # --- Step 2: download reference docs from GitHub, upload to your bucket -
    # Each file in KB_DOC_FILES is downloaded from GITHUB_RAW_BASE (public, no
    # AWS credentials needed) and uploaded into your bucket under
    # TARGET_KB_PREFIX, which the managed knowledge base then ingests.
    if not KB_DOC_FILES:
        raise RuntimeError(
            "KB_DOC_FILES is empty. List the reference document filenames to fetch "
            "from GITHUB_RAW_BASE in the Configuration section."
        )
    _uploaded = 0
    for _fname in KB_DOC_FILES:
        _url = f"{GITHUB_RAW_BASE}/{_fname}"
        try:
            with urllib.request.urlopen(_url) as _resp:
                if _resp.status != 200:
                    raise RuntimeError(f"HTTP {_resp.status} for {_url}")
                _body = _resp.read()
        except Exception as _dl_exc:
            raise RuntimeError(
                f"Failed to download reference document from {_url}: {_dl_exc}. "
                "Confirm GITHUB_RAW_BASE and KB_DOC_FILES point at files that are "
                "pushed and public in the repo."
            ) from _dl_exc
        _dst_key = f"{TARGET_KB_PREFIX}{_fname}"
        s3.put_object(Bucket=target_bucket, Key=_dst_key, Body=_body)
        _uploaded += 1
    print(
        f"[2/6] Downloaded {_uploaded} document(s) from GitHub and uploaded to "
        f"s3://{target_bucket}/{TARGET_KB_PREFIX}"
    )

    # --- Step 3: create the IAM role the knowledge base assumes ------------
    _trust_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Principal": {"Service": "bedrock.amazonaws.com"},
                "Action": "sts:AssumeRole",
                "Condition": {"StringEquals": {"aws:SourceAccount": account_id}},
            }
        ],
    }
    try:
        _role = iam.create_role(
            RoleName=KB_ROLE_NAME,
            AssumeRolePolicyDocument=json.dumps(_trust_policy),
            Description="Role assumed by the CausalIF Auto MPG managed knowledge base.",
        )
        role_arn = _role["Role"]["Arn"]
        print(f"[3/6] Created IAM role: {KB_ROLE_NAME}")
    except ClientError as exc:
        if exc.response["Error"]["Code"] == "EntityAlreadyExists":
            role_arn = iam.get_role(RoleName=KB_ROLE_NAME)["Role"]["Arn"]
            print(f"[3/6] Reusing existing IAM role: {KB_ROLE_NAME}")
        else:
            raise

    # Inline policy: read the docs bucket + invoke Bedrock models (embeddings /
    # managed models used by the knowledge base).
    _kb_policy = {
        "Version": "2012-10-17",
        "Statement": [
            {
                "Effect": "Allow",
                "Action": ["s3:GetObject", "s3:ListBucket"],
                "Resource": [
                    f"arn:aws:s3:::{target_bucket}",
                    f"arn:aws:s3:::{target_bucket}/*",
                ],
            },
            {
                "Effect": "Allow",
                "Action": ["bedrock:InvokeModel", "bedrock:Rerank"],
                "Resource": "*",
            },
        ],
    }
    iam.put_role_policy(
        RoleName=KB_ROLE_NAME,
        PolicyName="CausalIFAutoMpgKBAccess",
        PolicyDocument=json.dumps(_kb_policy),
    )
    # IAM role propagation is eventually consistent; give it a moment before
    # Bedrock tries to assume it.
    time.sleep(10)

    # --- Step 4: create the managed knowledge base (idempotent by name) ----
    def _find_kb_by_name(name):
        _p = bedrock_agent.get_paginator("list_knowledge_bases")
        for _pg in _p.paginate():
            for _s in _pg.get("knowledgeBaseSummaries", []):
                if _s.get("name") == name:
                    return _s["knowledgeBaseId"]
        return None

    kb_id = _find_kb_by_name(KB_NAME)
    if kb_id:
        print(f"[4/6] Reusing existing managed knowledge base: {kb_id}")
    else:
        _kb = bedrock_agent.create_knowledge_base(
            name=KB_NAME,
            roleArn=role_arn,
            description="CausalIF Auto MPG demo managed knowledge base.",
            knowledgeBaseConfiguration={
                "type": "MANAGED",
                "managedKnowledgeBaseConfiguration": {"embeddingModelType": "MANAGED"},
            },
        )
        kb_id = _kb["knowledgeBase"]["knowledgeBaseId"]
        print(f"[4/6] Created managed knowledge base: {kb_id}")

    # Wait for the knowledge base to leave CREATING before adding a data source.
    for _ in range(60):
        _status = bedrock_agent.get_knowledge_base(knowledgeBaseId=kb_id)[
            "knowledgeBase"
        ]["status"]
        if _status in ("ACTIVE", "AVAILABLE"):
            break
        if _status == "FAILED":
            raise RuntimeError(f"Knowledge base {kb_id} entered FAILED state.")
        time.sleep(10)

    # --- Step 5: attach S3 data source + run ingestion (idempotent) --------
    def _find_data_source(kbid, name):
        _p = bedrock_agent.get_paginator("list_data_sources")
        for _pg in _p.paginate(knowledgeBaseId=kbid):
            for _s in _pg.get("dataSourceSummaries", []):
                if _s.get("name") == name:
                    return _s["dataSourceId"]
        return None

    _ds_name = "auto-mpg-docs"
    ds_id = _find_data_source(kb_id, _ds_name)
    if ds_id:
        print(f"[5/6] Reusing existing data source: {ds_id}")
    else:
        _ds = bedrock_agent.create_data_source(
            knowledgeBaseId=kb_id,
            name=_ds_name,
            dataSourceConfiguration={
                "type": "S3",
                "s3Configuration": {
                    "bucketArn": f"arn:aws:s3:::{target_bucket}",
                    "inclusionPrefixes": [TARGET_KB_PREFIX],
                },
            },
        )
        ds_id = _ds["dataSource"]["dataSourceId"]
        print(f"[5/6] Created data source: {ds_id}")

    # Start an ingestion (sync) job and wait for it to complete so the docs are
    # queryable before the analysis runs.
    _job = bedrock_agent.start_ingestion_job(knowledgeBaseId=kb_id, dataSourceId=ds_id)
    _job_id = _job["ingestionJob"]["ingestionJobId"]
    print(f"[6/6] Started ingestion job {_job_id}; waiting for completion...")
    for _ in range(60):
        _js = bedrock_agent.get_ingestion_job(
            knowledgeBaseId=kb_id, dataSourceId=ds_id, ingestionJobId=_job_id
        )["ingestionJob"]["status"]
        if _js == "COMPLETE":
            print("      Ingestion complete.")
            break
        if _js == "FAILED":
            raise RuntimeError(f"Ingestion job {_job_id} FAILED.")
        time.sleep(10)

    return kb_id


# Only bootstrap when asked AND when a KB id has not already been provided.
if not RUN_BOOTSTRAP:
    print(
        "Automated setup skipped: RUN_BOOTSTRAP is False. Using KNOWLEDGE_BASE_ID "
        "as configured (blank means the notebook runs without a retriever tool)."
    )
elif isinstance(KNOWLEDGE_BASE_ID, str) and KNOWLEDGE_BASE_ID.strip():
    print(
        f"Automated setup skipped: KNOWLEDGE_BASE_ID is already set "
        f"('{KNOWLEDGE_BASE_ID}'). Delete it (set to \"\") to re-provision."
    )
else:
    try:
        KNOWLEDGE_BASE_ID = _bootstrap_managed_kb()
        print(f"\nAutomated setup complete. KNOWLEDGE_BASE_ID = '{KNOWLEDGE_BASE_ID}'.")
    except Exception as exc:  # noqa: BLE001 - surface the failed setup step clearly
        print(
            "ERROR - Automated setup failed while creating the S3 bucket, fetching "
            "the reference documents from GitHub, or provisioning the managed "
            "knowledge base. "
            "Confirm the execution role has the permissions listed in this "
            "section's description and that AWS_REGION supports managed knowledge "
            "bases (e.g. us-west-2)."
        )
        print(f"       Details: {type(exc).__name__}: {exc}")
        raise

### Retriever tool (Amazon Bedrock managed knowledge base)

This cell builds the retriever tool that lets CausalIF query the Amazon Bedrock knowledge base for domain context. It reads only the `KNOWLEDGE_BASE_ID`, `KB_NUM_RESULTS`, `RETRIEVER_TOOL_NAME`, and `RETRIEVER_TOOL_DESCRIPTION` values from the Config_Cell (the id is normally filled in by the Section 0 bootstrap above) — nothing is hardcoded here.

Because this demo uses a **managed** knowledge base, the retriever calls the Bedrock `bedrock-agent-runtime` `Retrieve` API directly with `managedSearchConfiguration` (managed knowledge bases require this instead of the `vectorSearchConfiguration` used by customer-managed knowledge bases) and wraps the result as a LangChain `Tool`. Calling `Retrieve` directly keeps the notebook working with the pinned `langchain-aws` version rather than depending on a newer release's managed-KB support. Credentials for the `Retrieve` call are taken implicitly from the SageMaker Studio execution role.

When `KNOWLEDGE_BASE_ID` is left blank (bootstrap disabled and no id provided), this cell binds `retriever_tool = None` and prints a note; the engine-configuration cell then runs CausalIF without a retriever, relying on the model's background knowledge alone.

**Expected output:** a confirmation line naming the knowledge base id the retriever tool was built for, or a note that the retriever tool was skipped because no `KNOWLEDGE_BASE_ID` is set. If the id is set but the retriever cannot be built, this cell prints a step-identifying error and re-raises so the misconfiguration is fixed before the analysis runs.

In [ ]:
# Build the CausalIF retriever tool for the MANAGED knowledge base from the
# Config_Cell settings. Binds `retriever_tool`, consumed by the engine-config
# cell. Managed knowledge bases require managedSearchConfiguration on Retrieve,
# so we call bedrock-agent-runtime directly and wrap it as a LangChain Tool
# (works with the pinned langchain-aws version).

# Always bind retriever_tool so the downstream engine-config cell can reference
# it unconditionally (it stays None when no knowledge base is configured).
retriever_tool = None

if not (isinstance(KNOWLEDGE_BASE_ID, str) and KNOWLEDGE_BASE_ID.strip()):
    # No knowledge base configured: run without a retriever tool. This is a
    # supported mode (the model uses its background knowledge alone), so we
    # only note it rather than treating it as an error.
    print(
        "Retriever tool skipped: KNOWLEDGE_BASE_ID is blank, so CausalIF will run "
        "without a Knowledge Base retriever tool (model background knowledge only)."
    )
else:
    try:
        # bedrock-agent-runtime resolves AWS credentials implicitly from the
        # SageMaker execution role; no access keys are read here.
        _kb_runtime = boto3.client("bedrock-agent-runtime", region_name=AWS_REGION)

        def _retrieve_from_kb(query: str) -> str:
            """Query the managed knowledge base and return concatenated passages.

            Uses managedSearchConfiguration (required for managed knowledge
            bases; vectorSearchConfiguration is customer-managed KBs only).
            """
            _resp = _kb_runtime.retrieve(
                knowledgeBaseId=KNOWLEDGE_BASE_ID,
                retrievalQuery={"text": query},
                retrievalConfiguration={
                    "managedSearchConfiguration": {
                        "numberOfResults": KB_NUM_RESULTS,
                    }
                },
            )
            _passages = []
            for _r in _resp.get("retrievalResults", []):
                _text = (_r.get("content") or {}).get("text")
                if _text:
                    _passages.append(_text)
            # Join passages into a single string the LLM tool interface expects.
            return "\n\n".join(_passages) if _passages else "No relevant documents found."

        # Wrap the retrieve function as a LangChain Tool CausalIF can call. The
        # name and description come from the Config_Cell so the LLM knows what
        # the knowledge base holds and when to query it.
        retriever_tool = Tool(
            name=RETRIEVER_TOOL_NAME,
            description=RETRIEVER_TOOL_DESCRIPTION,
            func=_retrieve_from_kb,
        )
        print(
            f"Retriever tool created for managed Knowledge Base "
            f"'{KNOWLEDGE_BASE_ID}' (numberOfResults={KB_NUM_RESULTS})."
        )
    except Exception as exc:  # noqa: BLE001 - surface a KB misconfiguration clearly
        retriever_tool = None
        print(
            "ERROR - Retriever tool setup failed: could not build the Retrieve "
            f"tool for KNOWLEDGE_BASE_ID '{KNOWLEDGE_BASE_ID}'. Confirm the id is "
            "correct, that it is a managed knowledge base, and that the execution "
            "role may call bedrock:Retrieve against it. Set "
            "KNOWLEDGE_BASE_ID to \"\" (and RUN_BOOTSTRAP=False) to run without a "
            "retriever tool."
        )
        print(f"       Details: {type(exc).__name__}: {exc}")
        raise

## 3. Data acquisition

This step fetches the UCI Auto MPG dataset (dataset id 9) from the UCI Machine Learning Repository using the official `ucimlrepo` client. The features and target returned by the client are concatenated into a single DataFrame, `raw_df`, and whatever columns come back are normalized to consistent snake_case names (for example `model year` -> `model_year`, `car name` -> `car_name`) so they are valid identifiers and match CausalIF's target extraction. The client may return **8 or 9 columns** depending on whether the free-text `car_name` column is included (it carries an ID/other role rather than a feature/target role, so it can be omitted); the cell does not assume a fixed count. Instead it requires only the seven columns the analysis actually needs — `mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year` — while `car_name` and `origin` are optional.

**Expected output:** the dataset shape printed as rows x columns (roughly 398 rows and 8 or 9 columns) followed by the first 5 records of `raw_df`. If the fetch fails, times out, or any required analysis column is missing after normalization, this cell prints an identifying error and leaves `raw_df` unbound so downstream cells do not run against partial data.

In [ ]:
# Fetch the UCI Auto MPG dataset (id 9) and assemble raw_df.
# The fetch is wrapped so any failure/timeout prints an identifying error and
# leaves no partial raw_df bound for downstream cells.

# ucimlrepo returns features + targets for dataset 9. Depending on how the
# repository classifies each variable, the assembled frame has 8 or 9 columns:
# the free-text "car name" column carries an ID/other role (not feature/target)
# and may be omitted, so we do NOT assume a fixed column count. Instead we
# normalize whatever column names come back to snake_case (valid identifiers
# that let CausalIF match column tokens when extracting the query target) and
# then require only the columns the analysis actually needs.

# The 7 analysis columns that MUST be present after normalization. car_name and
# origin are optional here (the preparation cell excludes both), so their
# absence must NOT fail acquisition.
REQUIRED_COLUMNS = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
]


def _to_snake_case(col):
    """Normalize a raw UCI column name to a snake_case identifier.

    Lowercases, trims surrounding whitespace, and turns spaces/hyphens into
    underscores so e.g. "model year" -> "model_year" and "car name" ->
    "car_name". Applied generically to every returned column, so no positional
    (fixed-count) assignment is needed.
    """
    return str(col).strip().lower().replace(" ", "_").replace("-", "_")


# Ensure no stale binding survives a failed fetch below.
raw_df = None

try:
    auto_mpg = fetch_ucirepo(id=9)
    # ucimlrepo returns features and target as separate DataFrames; concatenate
    # them column-wise into a single observational dataframe.
    _raw_df = pd.concat([auto_mpg.data.features, auto_mpg.data.targets], axis=1)

    # Normalize the ACTUAL returned column names to snake_case rather than
    # assuming a fixed count/order.
    _raw_df.columns = [_to_snake_case(col) for col in _raw_df.columns]

    # Validate by NAME: every required analysis column must be present. Missing
    # required columns halt with a descriptive error naming exactly which ones
    # are absent; optional car_name/origin do not trigger a failure.
    _missing = [col for col in REQUIRED_COLUMNS if col not in _raw_df.columns]
    if _missing:
        raise ValueError(
            f"Auto MPG data acquisition: required analysis columns are missing "
            f"after snake_case normalization: {_missing}. Present columns: "
            f"{list(_raw_df.columns)}."
        )

    # Bind raw_df only after the fetch and column normalization fully succeed.
    raw_df = _raw_df
except Exception as exc:  # noqa: BLE001 - surface any fetch/parse failure clearly
    # Leave no partial raw_df bound for downstream cells.
    raw_df = None
    print(
        "ERROR - Auto MPG data acquisition failed: could not retrieve the UCI "
        "Auto MPG dataset (id 9) from the UCI Machine Learning Repository."
    )
    print(f"       Details: {type(exc).__name__}: {exc}")
else:
    # Report dataset shape (rows x columns) and the first 5 records.
    print(f"Auto MPG dataset loaded: {raw_df.shape[0]} rows x {raw_df.shape[1]} columns")
    print("First 5 records:")
    display(raw_df.head())

## 4. Data preparation

This step cleans `raw_df` and selects the columns CausalIF will analyze, producing the observational DataFrame `prepared_df`. The `horsepower` column arrives with `"?"` sentinels for missing values, so it is coerced to a numeric type (turning those sentinels into `NaN`). The analysis is then restricted to the continuous numeric vehicle attributes (`ANALYSIS_COLUMNS`), deliberately excluding the free-text `car_name` and the categorical `origin` region code. Any record with a missing value in an analysis column is dropped, and the notebook confirms the prepared dataset still holds at least `MIN_SAMPLES` records before it is used for causal discovery. A `FACTOR_DESCRIPTIONS` text block defining each analyzed column is also assembled here for the CausalIF LLM to reason about domain semantics.

**Expected output:** the retained record count (the Auto MPG dataset yields 392 complete records after the 6 `"?"` horsepower rows are dropped, comfortably above the 100-sample minimum) and the list of analysis columns. If fewer than `MIN_SAMPLES` records remain, this cell prints a descriptive error naming the insufficient count and does not bind data for the engine.

In [ ]:
# Data preparation: clean raw_df and select the columns for causal analysis,
# producing prepared_df, ANALYSIS_COLUMNS, and FACTOR_DESCRIPTIONS.

# Coerce horsepower to numeric: the UCI Auto MPG data uses "?" as a missing-value
# sentinel, so errors="coerce" converts those non-numeric entries into NaN.
raw_df["horsepower"] = pd.to_numeric(raw_df["horsepower"], errors="coerce")

# Columns selected for causal analysis. car_name is excluded (free-text label,
# not a measurement). origin is excluded because it is a categorical region
# code (1 = US, 2 = Europe, 3 = Japan) rather than a continuous numeric
# measurement, so treating it as a numeric variable would be misleading.
ANALYSIS_COLUMNS = [
    "mpg",
    "cylinders",
    "displacement",
    "horsepower",
    "weight",
    "acceleration",
    "model_year",
]

# Restrict to the analysis columns and drop any record with a missing value in
# any of them, so the prepared dataset is fully numeric and complete.
prepared_df = raw_df[ANALYSIS_COLUMNS].dropna()

# Gate on the minimum sample count read from the Config_Cell. If too few
# records remain, report a descriptive error and do NOT bind data for the
# engine (leave prepared_df unusable for the downstream engine-config cell).
if len(prepared_df) < MIN_SAMPLES:
    print(
        f"ERROR - Data preparation: only {len(prepared_df)} records remain after "
        f"dropping rows with missing analysis-column values, which is fewer than "
        f"the required minimum of {MIN_SAMPLES}. The prepared dataset will NOT be "
        f"passed to CausalIF."
    )
    prepared_df = None
else:
    print(f"Retained {len(prepared_df)} records after cleaning.")
    print(f"Columns selected for analysis: {ANALYSIS_COLUMNS}")

# Factor descriptions passed to CausalIF so its LLM can reason about the meaning
# of each analyzed vehicle attribute. One "- column: definition" line per column.
FACTOR_DESCRIPTIONS = (
    "- mpg: fuel efficiency in miles per gallon (higher = more efficient)\n"
    "- cylinders: number of engine cylinders\n"
    "- displacement: engine displacement in cubic inches\n"
    "- horsepower: engine power output in horsepower\n"
    "- weight: vehicle weight in pounds\n"
    "- acceleration: time in seconds to accelerate 0-60 mph\n"
    "- model_year: model year of the vehicle (e.g. 70 = 1970)"
)

## 5. Causal analysis

This section configures the CausalIF engine and then runs causal discovery. This first cell handles **engine configuration**: it creates the Amazon Bedrock LLM that CausalIF uses (a LangChain `ChatBedrockConverse` built from the `BEDROCK_MODEL_ID` and `AWS_REGION` defined in the Configuration section) and then calls `set_causalif_engine` to hand the prepared observational data, the automotive-fuel-economy domains, the factor descriptions, and the bootstrap-stability parameters to the engine. When interventional analysis is enabled in the Config_Cell, causal (do-calculus) estimation is switched on here too. The next cell then runs the discovery query against this configured engine.

Credentials for Bedrock are taken implicitly from the SageMaker Studio execution role — no keys are read or printed anywhere. If the model cannot be created or `set_causalif_engine` fails (for example, the execution role lacks access to the model in the configured region, or credentials cannot be resolved), this cell prints a descriptive, step-identifying error and re-raises so the notebook halts before the analysis cell runs, while any earlier cell outputs are preserved.

**Expected output:** a confirmation line naming the AWS region and Bedrock model id that the engine was configured with.

In [ ]:
# Section 5 - Engine configuration.
# Build the Bedrock LLM from Config_Cell values and configure the CausalIF
# engine with the prepared data, domains, factor descriptions, and bootstrap
# parameters. Credentials come implicitly from the SageMaker execution role;
# no secrets are read or printed. Any failure prints a step-identifying error
# and re-raises so the notebook halts before the analysis cell runs.

# Validate the required configuration variables (blank/missing halts here,
# naming the offending variable) before touching Bedrock.
_require("BEDROCK_MODEL_ID", BEDROCK_MODEL_ID)
_require("AWS_REGION", AWS_REGION)

# The prepared dataset must exist (the preparation cell sets prepared_df to
# None if the sample-count gate failed); refuse to configure the engine
# without valid data.
if prepared_df is None:
    raise RuntimeError(
        "Engine configuration: prepared_df is not available (data preparation "
        "did not produce a valid dataset). Re-run the data preparation cell "
        "and ensure it retained at least MIN_SAMPLES records before configuring "
        "the CausalIF engine."
    )

# Domains give the LLM the correct background knowledge for these variables.
CAUSAL_DOMAINS = ["automotive", "fuel_economy", "vehicle_engineering"]

try:
    # temperature=0.0 for run-to-run stability. region_name selects the
    # Bedrock region; credentials are resolved implicitly from the execution
    # role (no access keys anywhere).
    bedrock_model = ChatBedrockConverse(
        model_id=BEDROCK_MODEL_ID,
        temperature=0.0,
        region_name=AWS_REGION,
    )

    # Hand all quantitative inputs to the engine. enable_causal_estimate is
    # driven by the Config_Cell ENABLE_INTERVENTION flag; there is no separate
    # fit call - the analysis runs later by calling causalif(query).
    set_causalif_engine(
        model=bedrock_model,
        dataframe=prepared_df,
        retriever_tool=retriever_tool,  # None when no KNOWLEDGE_BASE_ID is set
        domains=CAUSAL_DOMAINS,
        factor_descriptions=FACTOR_DESCRIPTIONS,
        bootstrap_iterations=BOOTSTRAP_ITERATIONS,
        bootstrap_threshold=BOOTSTRAP_THRESHOLD,
        enable_causal_estimate=ENABLE_INTERVENTION,
        # Keep Bedrock call concurrency low: high values trigger ServiceUnavailableException
        # (throttling) on the Converse API, which can empty the discovered causal graph.
        max_parallel_queries=2,
    )
except Exception as exc:  # noqa: BLE001 - surface the failed config step clearly
    # Identify the failed configuration step. Prior cell outputs are preserved
    # because we only print and re-raise (no state is cleared).
    print(
        "ERROR - CausalIF engine configuration failed while creating the "
        "Bedrock model or calling set_causalif_engine. Causal estimation "
        "remains disabled and the analysis cell will not run."
    )

    # Inspect the exception text for known AWS access / credential signals and
    # print the corresponding guidance before re-raising. Matching is on the
    # message text only; no secret values are read or printed.
    _msg = f"{type(exc).__name__}: {exc}"
    _lower = _msg.lower()

    _access_signals = (
        "accessdenied",
        "access denied",
        "don't have access",
        "do not have access",
        "not authorized",
        "could not be found",
        "resourcenotfound",
        "validationexception",
        "model access",
        "invocation of model id",
        "is not available",
    )
    _credential_signals = (
        "unable to locate credentials",
        "nocredentials",
        "credential",
        "unrecognizedclient",
        "invalid security token",
        "expiredtoken",
        "token has expired",
        "could not connect to the endpoint",
    )

    if any(signal in _lower for signal in _credential_signals):
        print(
            "       Authentication to Bedrock could not be established: the AWS "
            "credentials from the SageMaker Studio execution role could not be "
            f"resolved for region '{AWS_REGION}'. Confirm the notebook is running "
            "with a valid execution role."
        )
    elif any(signal in _lower for signal in _access_signals):
        print(
            f"       Amazon Bedrock model access is unavailable in region "
            f"'{AWS_REGION}' for model '{BEDROCK_MODEL_ID}'. Models auto-enable on "
            "first use, so confirm the execution role has bedrock:InvokeModel for "
            "this model and is not blocked by an IAM policy or SCP. For Anthropic "
            "models, a first-time user may need to submit use-case details in the "
            "Bedrock console before the first call. Otherwise set a different "
            "BEDROCK_MODEL_ID / AWS_REGION in the Configuration section."
        )

    print(f"       Details: {_msg}")
    raise
else:
    # On success, confirm the engine is configured and name the region + model.
    print(
        f"CausalIF engine configured (region='{AWS_REGION}', "
        f"model='{BEDROCK_MODEL_ID}')."
    )

In [ ]:
# Section 5 - Causal discovery.
# Run CausalIF against the engine configured above, using the supported
# "factors influencing <target>" query form with mpg as the target. The
# returned dict is stored in `result` for the results-presentation section.
#
# causalif() catches errors internally and returns {"success": False, ...}
# rather than raising, so we check result["success"] and, on failure, print a
# descriptive step-identifying error and raise to halt. This stops the
# presentation cells from running against a failed/empty result.
result = causalif("factors influencing mpg")

if not result.get("success", False):
    # Read the error text defensively: summary is usually present, and
    # algorithm_details / its "error" key may be absent depending on where the
    # analysis failed.
    _summary = result.get("summary")
    _algo_details = result.get("algorithm_details") or {}
    _algo_error = _algo_details.get("error") if isinstance(_algo_details, dict) else None
    _detail = _algo_error or _summary or "no error detail was provided by CausalIF."
    print(
        "ERROR - CausalIF analysis failed: the causal-discovery query "
        "'factors influencing mpg' did not complete successfully. The results "
        "presentation section will not run against this failed result."
    )
    print(f"       Details: {_detail}")
    raise RuntimeError(f"CausalIF analysis failed: {_detail}")

print(
    f"CausalIF analysis complete for target "
    f"'{result.get('target_factor', 'mpg')}'."
)

## 6. Results presentation

This final section presents the causal-discovery `result` in three complementary ways so the business question — *What factors influence MPG in cars?* — is answered clearly:

1. **Causal graph** — a rendered diagram of the discovered causal structure (this cell), showing the vehicle attributes and the directed edges between them.
2. **Edge table** — a tabular listing of every causal edge with its source, target, direction, and effect size, for readers who want the exact numbers behind the diagram.
3. **Written summary and answer** — a plain-language rundown of the factors found to influence MPG (and, when interventional analysis is enabled, an estimated effect of setting one factor to the configured level), followed by a concluding statement that answers the business question.

This first cell renders the causal graph with `visualize_causalif_results`. If the visualization fails, it prints a descriptive, step-identifying error but does **not** halt the notebook, so the edge table, written summary, and concluding answer below still run.

**Expected output:** an interactive causal-graph figure with `mpg` and the analyzed vehicle attributes as nodes and the discovered causal relationships as directed edges. The cell enlarges the figure and bumps the edge-label font for readability, saves a responsive interactive HTML copy (named by `OUTPUT_HTML`) next to the notebook so it can be opened and zoomed in a browser, and then calls `fig.show()` to display the figure inline.

In [ ]:
# Section 6 - Causal graph visualization.
# Render the discovered causal graph from `result`. A visualization failure is
# caught and reported (print, no re-raise) so the remaining presentation cells
# (edge table, written summary, concluding answer) still run.

try:
    fig = visualize_causalif_results(result)

    # Enlarge the canvas so nodes spread out and converging arrowheads don't overlap.
    # (visualize_causalif_results takes no layout args, but it returns a plain Plotly
    #  figure, so we size it here with Plotly's own API.)
    fig.update_layout(width=1400, height=1000)

    # Bump the edge-label font. Edge labels (e.g. 'P=0.68↓') are the pure text trace;
    # node labels are the markers+text trace, so this selector leaves node names as-is.
    fig.update_traces(textfont_size=18, selector=dict(mode="text"))

    # Save a responsive HTML copy that fills the browser window (handy for zooming in
    # on the arrowheads).
    fig.write_html(
        str(OUTPUT_HTML),
        include_plotlyjs="cdn",
        full_html=True,
        default_width="100%",
        default_height="100vh",
        config={"responsive": True},
    )
    print(f"Saved interactive graph to: {OUTPUT_HTML}")
    fig.show()
except Exception as exc:  # noqa: BLE001 - report and continue, do not abort
    print(
        "ERROR - Results presentation: rendering the causal graph with "
        "visualize_causalif_results failed. Skipping the graph and continuing "
        "with the edge table, written summary, and concluding answer below."
    )
    print(f"       Details: {type(exc).__name__}: {exc}")

In [ ]:
# Section 6 - Causal edge table.
# Build a table listing every causal edge with its source, target, direction,
# and effect size, so a reader can read the exact relationships without
# interpreting the diagram.
#
# Schema note: result["causal_graph"]["edges"] is a list of 3-tuples
# (source, target, attrs). The attrs dict may contain "do_direction" (the edge
# direction label, e.g. "positive"/"negative") and "do_probability" (the ATE /
# effect-size magnitude). Neither is guaranteed: the engine prunes zero-ATE
# edges and flags failed-ATE edges as undirected, so the builder reads both
# defensively and uses an empty placeholder rather than raising. Older library
# code can emit length-2 (source, target) tuples, which are tolerated by
# treating attrs as {}.

EDGE_TABLE_COLUMNS = ["from", "to", "direction", "effect_size"]

# Read result["causal_graph"]["edges"] defensively: either key may be absent,
# in which case there are simply no edges to display.
_causal_graph = result.get("causal_graph") or {}
_edges = _causal_graph.get("edges") if isinstance(_causal_graph, dict) else None
if not _edges:
    print("There are no causal edges to display.")
else:
    _rows = []
    for _edge in _edges:
        # edge[0] = source, edge[1] = target, edge[2] = attrs (may be absent).
        _source = _edge[0]
        _target = _edge[1]
        _attrs = _edge[2] if len(_edge) > 2 and isinstance(_edge[2], dict) else {}

        # direction: the do_direction label, empty placeholder if absent.
        _direction = _attrs.get("do_direction")
        if _direction is None:
            _direction = ""

        # effect_size: the do_probability (ATE) formatted to 2 decimals. Use an
        # empty placeholder if it is absent or not a number, without raising.
        _ate = _attrs.get("do_probability")
        try:
            _effect_size = f"{float(_ate):.2f}"
        except (TypeError, ValueError):
            _effect_size = ""

        _rows.append(
            {
                "from": _source,
                "to": _target,
                "direction": _direction,
                "effect_size": _effect_size,
            }
        )

    # Enforce exactly the four columns in the required order.
    edge_table_df = pd.DataFrame(_rows, columns=EDGE_TABLE_COLUMNS)
    print("Causal edges:")
    display(edge_table_df)

In [ ]:
# Section 6 - Written summary and plain-language answer.
# Read the factors CausalIF identified as influencing mpg from
# result["strongest_causal_influences"] (a list of dicts, each carrying an
# "influencing_factor" name). The key is read defensively: a missing/None value
# is treated as an empty list. For each factor we look up its edge into mpg in
# an edge-attribute map (built from result["causal_graph"]["edges"], which are
# (source, target, attrs) 3-tuples) and translate the edge's do_direction into
# plain language: positive -> increases MPG, negative -> decreases MPG, and
# neutral/missing/other -> influences MPG with an undetermined direction.

# Build a map from source factor -> its edge attributes for edges pointing into
# mpg, so each influencing factor's direction can be looked up. Read the edges
# defensively (either key may be absent) and tolerate length-2 (source, target)
# tuples by treating their attributes as {}.
_causal_graph = result.get("causal_graph") or {}
_edges = _causal_graph.get("edges") if isinstance(_causal_graph, dict) else None
_edges = _edges or []

_edge_attr_by_factor = {}
for _edge in _edges:
    _source = _edge[0]
    _target = _edge[1]
    _attrs = _edge[2] if len(_edge) > 2 and isinstance(_edge[2], dict) else {}
    # Only edges whose target is mpg describe a factor's influence on mpg.
    if _target == "mpg":
        _edge_attr_by_factor[_source] = _attrs


def _direction_phrase(do_direction):
    """Translate an edge do_direction label into a plain-language phrase."""
    if do_direction == "positive":
        return "increases MPG"
    if do_direction == "negative":
        return "decreases MPG"
    # neutral, "shift", missing, or any other value -> undetermined direction.
    return "influences MPG with an undetermined direction"


# Read the identified influences defensively (missing/None -> empty list).
_influences = result.get("strongest_causal_influences") or []

if not _influences:
    print("no influencing factors were identified")
    _identified_factors = []
else:
    _identified_factors = []
    print("Factors identified as influencing MPG:")
    for _influence in _influences:
        # Each influence is a dict; read its factor name defensively.
        _factor = _influence.get("influencing_factor") if isinstance(_influence, dict) else None
        if not _factor:
            continue
        _identified_factors.append(_factor)
        _attrs = _edge_attr_by_factor.get(_factor, {})
        _phrase = _direction_phrase(_attrs.get("do_direction"))
        print(f"- {_factor}: {_phrase}")

# One-line plain-language answer naming the identified factors. The concluding
# markdown cell frames these runtime-derived names when answering the business
# question.
if _identified_factors:
    _factor_list = ", ".join(_identified_factors)
    _answer = (
        f"Answer: the factors found to influence MPG are: {_factor_list}."
    )
else:
    _answer = (
        "Answer: CausalIF identified no factors influencing MPG for this run."
    )
print()
print(_answer)

In [ ]:
# Section 6 - Interventional (do-calculus) query.
# When interventional analysis is enabled (ENABLE_INTERVENTION) AND at least
# one influencing factor was identified, run a do-calculus query estimating
# the effect on mpg of setting one identified factor to INTERVENTION_LEVEL,
# and print the returned summary. This requires enable_causal_estimate=True
# (set from ENABLE_INTERVENTION in the engine-config cell) and a completed
# discovery run, both of which hold at this point.
#
# Re-derive the identified factors defensively from result so this cell is
# robust if run standalone, rather than depending on _identified_factors from
# the written-summary cell.
_influences = result.get("strongest_causal_influences") or []
_intervention_factors = []
for _influence in _influences:
    _factor = _influence.get("influencing_factor") if isinstance(_influence, dict) else None
    if _factor:
        _intervention_factors.append(_factor)

# A failure here is caught and reported (print, no re-raise) so execution
# continues to the concluding cell (Requirement 8.9).
try:
    if not ENABLE_INTERVENTION:
        print(
            "Interventional query skipped: ENABLE_INTERVENTION is False in the "
            "Configuration section, so causal (do-calculus) estimation is disabled."
        )
    elif not _intervention_factors:
        print(
            "Interventional query skipped: no influencing factors were identified, "
            "so there is no factor to intervene on."
        )
    else:
        # Use the first identified factor as the intervention target.
        _intervention_factor = _intervention_factors[0]
        _intervention_query = (
            f"effect of setting {_intervention_factor} to {INTERVENTION_LEVEL} on mpg"
        )
        print(f"Interventional query: {_intervention_query}")
        _intervention_result = causalif_intervene(_intervention_query)
        # Read the summary defensively from the returned dict.
        _intervention_summary = (
            _intervention_result.get("summary")
            if isinstance(_intervention_result, dict)
            else None
        )
        if _intervention_summary:
            print(_intervention_summary)
        else:
            print(
                "The interventional query returned no summary text to display."
            )
except Exception as exc:  # noqa: BLE001 - report and continue, do not abort
    print(
        "ERROR - Results presentation: the interventional (do-calculus) query "
        "with causalif_intervene failed. Continuing with the concluding answer below."
    )
    print(f"       Details: {type(exc).__name__}: {exc}")

## Conclusion: What factors influence MPG in cars?

This notebook set out to answer one business question — **What factors influence MPG in cars?** — and the causal-discovery run above provides the answer.

**The factors that influence MPG are listed on the `Answer:` line printed by the written-summary cell above** (the line beginning `Answer: the factors found to influence MPG are: ...`). Because those factor names are derived at runtime from CausalIF's analysis of the Auto MPG data, they are reported there rather than hardcoded here, so the answer always reflects the current run. If that line instead reads that no factors were identified, then for this run CausalIF found no factor with a stable causal edge into MPG.

For each factor named on that `Answer:` line, the written summary also states the **direction** of its influence — whether it *increases MPG*, *decreases MPG*, or *influences MPG with an undetermined direction* — and the **edge table** gives the corresponding effect-size magnitude for every discovered causal edge. Read those two outputs together to see not just *which* attributes drive fuel efficiency but *how* each one pushes it.

When interventional analysis is enabled, the **interventional-query result** printed just above quantifies this further, estimating the effect on MPG of setting one of the identified factors to the configured intervention level — turning the causal structure into a concrete "what-if" answer.

Taken together, the causal graph, the edge table, the written summary's `Answer:` line, and the interventional result answer the business question directly: the vehicle attributes named above are the factors that causally influence a car's miles per gallon, in the directions and magnitudes reported.